In [1]:
import pandas as pd
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from imblearn.over_sampling import SMOTE
import os
import time

In [2]:
# Constants
NUM_CLASSES = 4
RANDOM_STATE = 1082
TRAIN_SIZE = 0.2
DATA_PATH = "../../../Datasets/excel/2dSpoofingDataset/GPS_Data_Simplified_2D_Feature_Map.xlsx"
MODEL_NAME = "Random Forest (SMOTE)"
DATASET_NAME = "GPS_Data_Simplified_2D_Feature_Map.xlsx"
RESULTS_CSV = "../../../2D_Results.csv"

In [3]:
# Load the dataset
try:
    df = pd.read_excel("../../../Datasets/excel/2dSpoofingDataset/GPS_Data_Simplified_2D_Feature_Map.xlsx")
except FileNotFoundError:
    print("Error: The file '../../../Datasets/excel/GPS_Data_Simplified_2D_Feature_Map.xlsx' was not found.")
    print("Please make sure the path is correct.")
    exit()

In [4]:
# Split data into features and target
X = df.drop('Output', axis=1)
y = df['Output']

print(f"\nFeatures (X) shape: {X.shape}")
print(f"Target (y) shape: {y.shape}")

# Split into train and test before SMOTE
X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=TRAIN_SIZE, random_state=RANDOM_STATE, stratify=y
)

print("\nApplying SMOTE to the training data...")
count_of_majority_class = y_train.value_counts()[0]
print(f"Number of samples in the majority class (0) in y_train: {count_of_majority_class}")
desired_strategy = {
    1: count_of_majority_class,
    2: count_of_majority_class,
    3: count_of_majority_class
}

smote = SMOTE(random_state=RANDOM_STATE, sampling_strategy=desired_strategy)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print(f"Resampled training features shape: {X_train_resampled.shape}")
print(f"Resampled training target shape: {y_train_resampled.shape}")
print(f"Class distribution in resampled y_train:\n{y_train_resampled.value_counts(normalize=True)}")


Features (X) shape: (510530, 13)
Target (y) shape: (510530,)

Applying SMOTE to the training data...
Number of samples in the majority class (0) in y_train: 79565
Resampled training features shape: (318260, 13)
Resampled training target shape: (318260,)
Class distribution in resampled y_train:
Output
0    0.25
3    0.25
1    0.25
2    0.25
Name: proportion, dtype: float64


In [5]:
# Train the MLP Classifier model

model = model = RandomForestClassifier(
    n_estimators=200,
    n_jobs=-1,
    random_state=RANDOM_STATE
)

train_start_time = time.time()
model.fit(X_train_resampled, y_train_resampled)
train_time = time.time() - train_start_time

In [6]:
# Evaluate the model

print("\nEvaluating the model on the test set...")
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
print(f"Model Accuracy on Test Set: {accuracy * 100:.2f}%")
print(f"F1 Score: {f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))


Evaluating the model on the test set...
Model Accuracy on Test Set: 92.78%
F1 Score: 0.9308

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.94      0.96    318260
           1       0.76      0.87      0.81     29166
           2       0.67      0.81      0.73     35386
           3       0.93      0.97      0.95     25612

    accuracy                           0.93    408424
   macro avg       0.83      0.90      0.86    408424
weighted avg       0.94      0.93      0.93    408424



In [7]:
# Save results to CSV
results = [{
    "model": MODEL_NAME,
    "dataset": DATASET_NAME,
    "f1score": f1,
    "accuracy": accuracy,
    "train_time_sec": train_time,
    "parameters": str(model.get_params()),
    "num_epochs": "N/A"
}]
results_df = pd.DataFrame(results)

file_exists = os.path.isfile(RESULTS_CSV)
results_df.to_csv(
    RESULTS_CSV,
    mode='a',
    header=not file_exists,
    index=False
)
print(f"Results appended to {RESULTS_CSV}")

Results appended to ../../../2D_Results.csv
